# Лекција 13 - Меморија агента


## Постављање

Овај бележник демонстрира како изградити агента за резервацију путовања са **перзистентном меморијом** користећи **Microsoft Agent Framework** (MAF).

Нучићете како различите врсте меморије агента — радна, краткорочна и дугорочна — утичу на то како агент задржава и користи информације кроз разговоре.

**Предуслови:**
- Пројекат у Azure AI Foundry са постављеним моделом за ћаскање (нпр. `gpt-4o-mini`).
- Пријављени сте преко Azure CLI — покрените `az login` у вашем терминалу.
- `AZURE_AI_PROJECT_ENDPOINT` — ваш крајњи тачка пројекта у Azure AI Foundry.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — име вашег постављеног модела.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Типови памћења агента

AI агенти могу користити различите врсте памћења, од којих свака има своју посебну сврху:

### Радна меморија
Сам разговор — поруке размењене у једној сесији. Агент може да се врати на претходне поруке у истом разговору како би одржао кохеренцију. У MAF-у се ово креира помоћу **`agent.create_session()`**, што враћа `AgentSession`.

### Краткорочна меморија
Информације које трају током трајања задатка или сесије, али се не чувају трајно. На пример, агент може током вишекратног планирања прикупљати чињенице и користити их за израду коначног распореда.

### Дугорочна меморија
Преференције и чињенице које трају **преко сесија**. Повратни корисник не би требало да поново понавља своја дијетална ограничења или стил путовања. Дугорочна меморија је обично подржана спољним носачем — базом података, фајлом или векторским индексом — и представљена агенту преко алата.


## Радна меморија са сесијама

Најједноставнији облик меморије је сесија разговора. Када проследите исту сесију (креирану преко `agent.create_session()`) узастопним позивима `agent.run()`, агент види пуну историју тог разговора и може да се сети ранијих детаља.

Хајде да направимо туристичког агента и демонстрирамо радну меморију.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Агент је правилно запамтио буџет јер обе поруке деле исту сесију. Ово је **радна меморија** — постоји само током трајања сесије.

### Шта се дешава са новом нити?

Ако креирамо **нову** сесију, агент нема сећање на претходни разговор:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Образац дугорочне меморије

Да бисмо памтили корисничке преференције **преко сесија**, потребна нам је упорна меморија која постоји ван самог разговора. Агент приступа овој меморији преко **алата** — функција које може да позива да би сачувао и преузео информације.

Испод имплементирамо једноставну меморију у оперативној меморији (у продукцији бисте користили базу података или индекс вектора) и изложимо је као алате које агент може користити.

### Архитектура
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Сценарио 1 — Корисник који први пут резервише путовање за годишњицу

Сара посећује по први пут. Агенти треба да сачувају њене преференције путем алата и користе их за препоруку хотела.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Сценарио 2 — Сара се враћа недељама касније

Сара почиње **потпуно нову нит** (симулирајући нову сесију). Радна меморија је празна, али дугорочна продавница преференци и даље има њене информације. Агенат треба да их преузме и искористи за персонализацију препорука.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Сажетак

У овом лекцији сте научили три типа меморије агента и како их имплементирати помоћу Microsoft Agent Framework-а:

| Тип меморије | MAF механизам | Животни век |
|---|---|---|
| **Радна** | `agent.create_session()` | Један разговор |
| **Краткорочна** | Накупљени контекст унутар нити | Један задатак / сесија |
| **Дугорочна** | Спољна складишта приступна преко `@tool` функција | Између сесија |

### Кључни закључци
1. **`agent.create_session()`** обезбеђује радну меморију — агент види комплетну историју разговора унутар сесије.
2. **Нове сесије губе контекст** — без дугорочне меморије агент не може да се сетити претходних разговора.
3. **`@tool` функције** премошћују сегмент — омогућавају агенту да сачува и преузме информације из трајне складиште.
4. **Персонализација се побољшава током времена** — што више преференција је сачувано, боље су препоруке агента.

### Примена у стварном свету
- **Корисничка служба**: памти историју и преференције корисника
- **Лични асистенти**: одржавају контекст током дана или недеља
- **Здравство**: прати информације о пацијентима и њихове преференције
- **Е-трговина**: персонализована куповина базирана на историји

### Следећи кораци
- Заменити у меморији речник базом података или складиштем вектора (нпр. Azure AI Search)
- Додати истек меморије за информације осетљиве на време
- Изградити системе вишеструких агената са заједничком меморијом
- Истражити Cognee notebook за меморију поткрепљену графом знања


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Одрицање од одговорности**:
Овај документ је преведен помоћу сервиса за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, молимо вас да имате у виду да аутоматски преводи могу садржати грешке или нетачности. Изворни документ на његовом оригиналном језику сматра се ауторитетним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било какве неспоразуме или погрешне тумачења која могу настати коришћењем овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
